# INITIAL IMPORT

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline


In [2]:
import os
from src.config import Configuration


CONFIG = Configuration(

)

In [9]:
# input_path = CONFIG.mr_nf_structural
# output_path = CONFIG.mr_nf_tensors
# output_96_path = CONFIG.mr_nf_tensors_96
# modalities = ['T1', 'T1GD', 'T2', 'FLAIR']
# filter_visits = True

input_path = CONFIG.brats_path_structural
output_path = CONFIG.brats_tensors
output_96_path = CONFIG.brats_tensors_96
modalities = ['t2', 't1ce', 't1', 'flair']
filter_visits = False


from maikol_utils.file_utils import make_dirs

make_dirs([output_path, output_96_path])

[True, True]

In [10]:
import os
import nibabel as nib
import torch
from tqdm import tqdm

from maikol_utils.print_utils import print_warn
from src.config import Configuration

def convert_niigz_to_tensor(CONFIG: Configuration, filter_visits: bool = True):
    """
    Convert patient NIfTI files to tensors.

    Args:
        CONFIG: Project configuration object.
        filter_visits:
            - True: keep one folder per base patient ID, preferring *_11 over *_21.
            - False: parse all patient folders found in input_path.
    """
    selected_entries = []

    if filter_visits:
        # Collect candidate folders by base ID and prefer _11 over _21.
        selected_patients = {}

        for patient_id in os.listdir(input_path):
            patient_path = os.path.join(input_path, patient_id)

            if not os.path.isdir(patient_path):
                print_warn(f"Skipping {patient_path} as it is not a directory")
                continue

            if not (patient_id.endswith('_11') or patient_id.endswith('_21')):
                continue

            base_id, visit_suffix = patient_id.rsplit('_', 1)
            existing = selected_patients.get(base_id)

            if existing is None:
                selected_patients[base_id] = (patient_id, visit_suffix)
                continue

            # Keep _11 whenever present; only use _21 if _11 does not exist.
            _, existing_suffix = existing
            if existing_suffix == '21' and visit_suffix == '11':
                selected_patients[base_id] = (patient_id, visit_suffix)

        selected_entries = [
            (base_id, patient_id)
            for base_id, (patient_id, _) in selected_patients.items()
        ]
    else:
        # Parse every patient directory as-is.
        for patient_id in os.listdir(input_path):
            patient_path = os.path.join(input_path, patient_id)

            if not os.path.isdir(patient_path):
                print_warn(f"Skipping {patient_path} as it is not a directory")
                continue

            selected_entries.append((patient_id, patient_id))

    for output_id, patient_id in tqdm(selected_entries):
        patient_path = os.path.join(input_path, patient_id)

        # Create an empty tensor to hold all 4 modalities (Shape: 4, X, Y, Z)
        tensor_list = []

        for mod in modalities:
            file_path = os.path.join(patient_path, f"{patient_id}_{mod}.nii.gz")
            img_data = nib.load(file_path).get_fdata()
            tensor_list.append(torch.tensor(img_data, dtype=torch.float32))

        # Stack into a single 4D tensor and save
        stacked_tensor = torch.stack(tensor_list)
        torch.save(stacked_tensor, os.path.join(output_path, f"{output_id}.pt"))


convert_niigz_to_tensor(CONFIG, filter_visits=filter_visits)

⚠️Skipping ../data/BraTS/BraTS2021_Training_Data/.DS_Store as it is not a directory⚠️


  0%|          | 0/1251 [00:00<?, ?it/s]

100%|██████████| 1251/1251 [09:50<00:00,  2.12it/s]


# Rescale

In [ ]:
import os
import torch
import torch.nn.functional as F
from tqdm import tqdm

TARGET = 96

for fname in os.listdir(output_path):
    if not fname.endswith(".pt"):
        continue
    vol = torch.load(os.path.join(output_path, fname), weights_only=True)  # (4, H, W, D)
    vol = F.interpolate(
        vol.unsqueeze(0), size=(TARGET, TARGET, TARGET),
        mode="trilinear", align_corners=False
    ).squeeze(0)
    torch.save(vol, os.path.join(output_96_path, fname))
    print(f"saved {fname}  {vol.shape}")

saved BraTS2021_01559.pt  torch.Size([4, 96, 96, 96])
saved BraTS2021_00504.pt  torch.Size([4, 96, 96, 96])
saved BraTS2021_00796.pt  torch.Size([4, 96, 96, 96])
saved BraTS2021_00792.pt  torch.Size([4, 96, 96, 96])
saved BraTS2021_01547.pt  torch.Size([4, 96, 96, 96])
saved BraTS2021_01351.pt  torch.Size([4, 96, 96, 96])
saved BraTS2021_00124.pt  torch.Size([4, 96, 96, 96])
saved BraTS2021_00730.pt  torch.Size([4, 96, 96, 96])
saved BraTS2021_00495.pt  torch.Size([4, 96, 96, 96])
saved BraTS2021_01041.pt  torch.Size([4, 96, 96, 96])
saved BraTS2021_01180.pt  torch.Size([4, 96, 96, 96])
saved BraTS2021_00284.pt  torch.Size([4, 96, 96, 96])
saved BraTS2021_00347.pt  torch.Size([4, 96, 96, 96])
saved BraTS2021_00733.pt  torch.Size([4, 96, 96, 96])
saved BraTS2021_01237.pt  torch.Size([4, 96, 96, 96])
saved BraTS2021_00028.pt  torch.Size([4, 96, 96, 96])
saved BraTS2021_00371.pt  torch.Size([4, 96, 96, 96])
saved BraTS2021_01427.pt  torch.Size([4, 96, 96, 96])
saved BraTS2021_01098.pt  to